# 111 — CPTAC-3 Acquisition and Audit

## Objective

Audit the already acquired **CPTAC-3 / GDC** open processed RNA-seq STAR Counts and DNA-methylation SeSAMe beta-value resources as candidate external inputs for Phase 8 — Orthogonal Validation.

This is a **Phase 1 acquisition and technical/metadata audit**, not an external validation analysis. It does not score, refit, select, or reorient any frozen Phase 2–7 program and does not test cross-cohort biological replication.

## Frozen acquisition inventory (pre-audit expectation)

- RNA-seq: 2,968 GDC file UUIDs; STAR - Counts; open Gene Expression Quantification.
- Methylation: 2,610 GDC file UUIDs; SeSAMe Methylation Beta Estimation; open Methylation Beta Value.
- Canonical immutable payloads: data/raw/cptac3/{rnaseq_star_counts,methylation_sesame}/{file_uuid}/{provider_filename}.
- Provider manifests and initial API metadata: config/manifests/cptac3/.

The original acquisition was **source-complete within these GDC query filters**, not restricted to primary tumors. Primary tumors, normal tissue, hematologic samples, recurrent tumors, and metastatic samples must not be pooled without a prespecified eligibility rule.

## Main tasks

1. Validate the frozen acquisition manifests against the initial GDC metadata exports.
2. Audit local UUID-directory, file size, and provider MD5 integrity.
3. Retrieve and retain an exact-file GDC associated-entity snapshot, without modifying the original metadata JSON.
4. Link files to their directly associated aliquots and then to samples through the GDC case/biospecimen hierarchy.
5. Characterize unresolved/multiple associations, sample types, platform and lineage metadata, and case/sample overlap.
6. Produce deterministic audit indices, provisional candidate-primary file inventories, and source fingerprints for review.
7. Build a registry **proposal**, without silently modifying config/raw_data_registry.json.

## Methodological boundary

GDC cases.samples can contain *all* samples associated with a case; it must **not** be interpreted as the sample producing a given file. A sample is attributed to a file only via a direct associated-entity identifier reconciled against the corresponding aliquot in the nested sample hierarchy. Unresolved or multiply associated files are retained for audit, not assigned arbitrarily.

A prespecified **provisional primary-tumor eligibility flag** requires exactly one directly associated aliquot, exactly one uniquely resolved sample, and sample_type == Primary Tumor. This flag is **not** a finalized Phase 8 analysis cohort. Lineage/platform compatibility, file multiplicity, biologic independence, and downstream validation rules require explicit review.

Raw inputs remain immutable. This notebook does not freeze a final cohort or automatically register an acquired-and-used resource. Publication and cohort freeze are deferred until the audit outputs are reviewed.

## Execution

Run top to bottom in the project's Python 3.11.8 environment. GDC access is required to enrich association metadata *only when a valid local snapshot does not yet exist*. Full local payload MD5 verification reads approximately 67.4 GiB, but does not change the files.

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import hashlib
import json
import time
from copy import deepcopy
from datetime import datetime, timezone

import pandas as pd
import requests

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import (
    load_raw_data_registry,
    validate_raw_data_registry,
)

In [ ]:
# =============================================================================
# Authoritative CPTAC-3 source paths and fixed acquisition scope
# =============================================================================

CPTAC3_MANIFEST_DIR = Paths.config / "manifests" / "cptac3"
CPTAC3_RAW_DIR = Paths.root / "data" / "raw" / "cptac3"
AUDIT_OUTPUT_DIR = Paths.metadata

SOURCE_SPECS = {
    "rnaseq": {
        "manifest": CPTAC3_MANIFEST_DIR / "gdc_manifest_cptac3_rnaseq.txt",
        "metadata": CPTAC3_MANIFEST_DIR / "gdc_metadata_cptac3_rnaseq.json",
        "payload_dir": CPTAC3_RAW_DIR / "rnaseq_star_counts",
        "expected_n_files": 2968,
        "data_category": "Transcriptome Profiling",
        "data_type": "Gene Expression Quantification",
        "workflow_type": "STAR - Counts",
    },
    "methylation": {
        "manifest": CPTAC3_MANIFEST_DIR / "gdc_manifest_cptac3_methylation.txt",
        "metadata": CPTAC3_MANIFEST_DIR / "gdc_metadata_cptac3_methylation.json",
        "payload_dir": CPTAC3_RAW_DIR / "methylation_sesame",
        "expected_n_files": 2610,
        "data_category": "DNA Methylation",
        "data_type": "Methylation Beta Value",
        "workflow_type": "SeSAMe Methylation Beta Estimation",
    },
}

RUN_FULL_PAYLOAD_MD5 = True

source_paths = {
    f"{modality}.{kind}": spec[kind]
    for modality, spec in SOURCE_SPECS.items()
    for kind in ("manifest", "metadata", "payload_dir")
}
missing_sources = [
    key for key, path in source_paths.items() if not path.exists()
]
if missing_sources:
    raise FileNotFoundError(f"Missing CPTAC-3 sources: {missing_sources}")

for key, value in source_paths.items():
    print(f"{key}: {project_relative_path(value)}")

In [ ]:
# =============================================================================
# Load and validate the source-complete GDC manifests
# =============================================================================

manifests = {}

for modality, spec in SOURCE_SPECS.items():
    frame = pd.read_csv(spec["manifest"], sep="\t", dtype="string")
    required = {"id", "filename", "md5", "size", "state"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{modality}: missing manifest columns {required - set(frame.columns)}")
    if len(frame) != spec["expected_n_files"]:
        raise ValueError(f"{modality}: unexpected manifest file count: {len(frame)}")
    if frame["id"].isna().any() or frame["id"].duplicated().any():
        raise ValueError(f"{modality}: missing or repeated file UUIDs")
    if frame[["filename", "md5", "size"]].isna().any().any():
        raise ValueError(f"{modality}: incomplete manifest identity")
    if not frame["md5"].str.fullmatch(r"[0-9a-fA-F]{32}").all():
        raise ValueError(f"{modality}: invalid provider MD5")
    frame["size"] = pd.to_numeric(frame["size"], errors="raise").astype("int64")
    if not frame["size"].gt(0).all():
        raise ValueError(f"{modality}: invalid file sizes")
    manifests[modality] = frame.sort_values("id").reset_index(drop=True)
    print(f"{modality}: {len(frame):,} files, {frame['size'].sum() / 1024**3:.2f} GiB")

if set(manifests["rnaseq"]["id"]) & set(manifests["methylation"]["id"]):
    raise ValueError("GDC file UUID unexpectedly occurs in both modalities")

In [ ]:
# =============================================================================
# Load the preserved acquisition-time GDC metadata snapshots
# =============================================================================

source_hits = {}
source_metadata_sha256 = {}

for modality, spec in SOURCE_SPECS.items():
    with spec["metadata"].open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict) or not isinstance(payload.get("data"), dict):
        raise ValueError(f"{modality}: unexpected GDC metadata JSON envelope")
    hits = payload["data"].get("hits")
    if not isinstance(hits, list):
        raise ValueError(f"{modality}: missing GDC data.hits")
    if len(hits) != len(manifests[modality]):
        raise ValueError(f"{modality}: metadata-to-manifest count mismatch")
    if len({hit.get("file_id", hit.get("id")) for hit in hits}) != len(hits):
        raise ValueError(f"{modality}: duplicate metadata file IDs")
    source_hits[modality] = {
        str(hit.get("file_id") or hit.get("id")): hit for hit in hits
    }
    source_metadata_sha256[modality] = calculate_sha256(spec["metadata"])
    print(f"{modality}: {len(hits):,} preserved metadata records")

In [ ]:
# =============================================================================
# Reconcile file identities, workflow, project, and access
# =============================================================================

for modality, spec in SOURCE_SPECS.items():
    for row in manifests[modality].itertuples(index=False):
        record = source_hits[modality].get(row.id)
        if record is None:
            raise ValueError(f"{modality}: missing metadata for {row.id}")
        if (
            record.get("file_name") != row.filename
            or int(record.get("file_size", -1)) != row.size
            or str(record.get("md5sum", "")).lower() != row.md5.lower()
        ):
            raise ValueError(f"{modality}: mismatched file identity for {row.id}")
        if record.get("access", "").lower() != "open":
            raise ValueError(f"{modality}: non-open resource {row.id}")
        if (
            record.get("data_category") != spec["data_category"]
            or record.get("data_type") != spec["data_type"]
            or (record.get("analysis") or {}).get("workflow_type") != spec["workflow_type"]
        ):
            raise ValueError(f"{modality}: workflow/category discrepancy {row.id}")
        cases = record.get("cases") or []
        if not cases or any(
            (case.get("project") or {}).get("project_id") != "CPTAC-3"
            for case in cases
        ):
            raise ValueError(f"{modality}: missing or unexpected GDC project for {row.id}")
    print(f"{modality}: exact manifest / metadata identity and cohort filters PASS")

## Local acquisition integrity

File identity is governed by the GDC file UUID, original filename, provider byte size, and provider MD5 in the version-controlled manifest. The file count is not interpreted as a patient count.

This is the reproducible replacement for the earlier ad hoc PowerShell validation. Any missing file, unexpected UUID directory, size mismatch, or MD5 mismatch fails the audit; it is not silently repaired or downloaded again.

In [ ]:
# =============================================================================
# Audit local GDC UUID-folder structure and unexpected entries
# =============================================================================

for modality, spec in SOURCE_SPECS.items():
    root = spec["payload_dir"]
    expected = set(manifests[modality]["id"])
    observed = {item.name for item in root.iterdir() if item.is_dir()}
    unexpected_files = [item.name for item in root.iterdir() if not item.is_dir()]
    if expected != observed or unexpected_files:
        raise ValueError(
            f"{modality}: folder structure discrepancy: "
            f"missing={len(expected - observed)}, "
            f"unexpected={len(observed - expected)}, "
            f"root-level extras={unexpected_files[:5]}"
        )
    print(f"{modality}: {len(observed):,} expected GDC UUID directories")

In [ ]:
# =============================================================================
# Validate payload file size and provider MD5 (read-only)
# =============================================================================

def md5_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


payload_rows = []
for modality, spec in SOURCE_SPECS.items():
    for index, row in enumerate(manifests[modality].itertuples(index=False), start=1):
        folder = spec["payload_dir"] / row.id
        file_path = folder / row.filename
        others = [item.name for item in folder.iterdir() if item.name != row.filename]
        if others or not file_path.is_file():
            raise ValueError(
                f"{modality}: missing file or unexpected folder contents for {row.id}: "
                f"{others[:5]}"
            )
        observed_size = file_path.stat().st_size
        if observed_size != row.size:
            raise ValueError(f"{modality}: size mismatch for {row.id}")
        observed_md5 = md5_file(file_path) if RUN_FULL_PAYLOAD_MD5 else None
        if RUN_FULL_PAYLOAD_MD5 and observed_md5.lower() != row.md5.lower():
            raise ValueError(f"{modality}: MD5 mismatch for {row.id}")
        payload_rows.append({
            "modality": modality,
            "file_id": row.id,
            "filename": row.filename,
            "expected_size_bytes": row.size,
            "observed_size_bytes": observed_size,
            "provider_md5": row.md5.lower(),
            "observed_md5": observed_md5,
            "size_matches": True,
            "md5_matches": bool(RUN_FULL_PAYLOAD_MD5),
        })
        if index % 500 == 0 or index == len(manifests[modality]):
            print(f"{modality}: {index:,} / {len(manifests[modality]):,} files checked")

payload_validation = pd.DataFrame(payload_rows).sort_values(
    ["modality", "file_id"]
).reset_index(drop=True)
print("Full MD5 verification performed:", RUN_FULL_PAYLOAD_MD5)

## Direct GDC file–biospecimen associations

The existing GDC metadata exports were requested with nested cases and samples, but without the **associated_entities** field. The notebook therefore retrieves an additional, exact-file snapshot from the GDC Files API, in batches restricted to the UUIDs in the already verified manifests.

For each file, the authoritative direct GDC association is evaluated first. An aliquot UUID is then resolved through the full nested case → sample → portion → analyte → aliquot hierarchy retained in the initial metadata snapshot. Merely finding a sample under the same case is insufficient.

This API snapshot is cached as a separate version-controlled source export. If it already exists, no new API request is made. Provider query results cannot add or remove file UUIDs from the frozen acquisition universe.

In [ ]:
# =============================================================================
# Define exact-file GDC association snapshot retrieval
# =============================================================================

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
GDC_FIELDS = ",".join([
    "file_id",
    "associated_entities",
    "platform",
    "cases.case_id",
    "cases.primary_site",
    "cases.disease_type",
])
GDC_BATCH_SIZE = 200

association_paths = {
    modality: CPTAC3_MANIFEST_DIR / f"gdc_associated_entities_cptac3_{modality}.json"
    for modality in SOURCE_SPECS
}


def fetch_association_batch(session, file_ids):
    request_body = {
        "filters": {
            "op": "in",
            "content": {"field": "file_id", "value": file_ids},
        },
        "fields": GDC_FIELDS,
        "format": "JSON",
        "size": len(file_ids),
    }
    for attempt in range(3):
        try:
            response = session.post(
                GDC_FILES_URL, json=request_body, timeout=(20, 180)
            )
            response.raise_for_status()
            data = response.json()["data"]
            hits = data["hits"]
            if len(hits) != len(file_ids) or data["pagination"]["total"] != len(file_ids):
                raise ValueError("GDC batch count differs from exact requested scope")
            return hits
        except (requests.RequestException, ValueError, KeyError) as error:
            if attempt == 2:
                raise RuntimeError("GDC exact-file association query failed") from error
            time.sleep(2 ** attempt)
    raise RuntimeError("Unreachable GDC retry path")

In [ ]:
# =============================================================================
# Acquire or load complete direct-association snapshots
# =============================================================================

association_snapshots = {}
for modality, spec in SOURCE_SPECS.items():
    snapshot_path = association_paths[modality]
    expected_ids = sorted(manifests[modality]["id"].tolist())

    if snapshot_path.is_file():
        with snapshot_path.open("r", encoding="utf-8") as handle:
            snapshot = json.load(handle)
        print(f"{modality}: reusing local direct-association snapshot")
    else:
        fetched = []
        with requests.Session() as session:
            for start in range(0, len(expected_ids), GDC_BATCH_SIZE):
                subset = expected_ids[start:start + GDC_BATCH_SIZE]
                fetched.extend(fetch_association_batch(session, subset))
                print(f"{modality}: GDC associations {len(fetched):,} / {len(expected_ids):,}")
        snapshot = {
            "provider": "NCI Genomic Data Commons",
            "endpoint": GDC_FILES_URL,
            "fields": GDC_FIELDS.split(","),
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "manifest_sha256": calculate_sha256(spec["manifest"]),
            "hits": sorted(fetched, key=lambda item: item.get("file_id", item.get("id", ""))),
        }

    if snapshot.get("manifest_sha256") != calculate_sha256(spec["manifest"]):
        raise ValueError(f"{modality}: direct-association snapshot manifest fingerprint differs")
    hits = snapshot.get("hits")
    if not isinstance(hits, list):
        raise ValueError(f"{modality}: invalid associated-entity snapshot")
    observed_ids = [str(hit.get("file_id") or hit.get("id")) for hit in hits]
    if len(observed_ids) != len(expected_ids) or sorted(observed_ids) != expected_ids:
        raise ValueError(f"{modality}: snapshot file IDs do not equal frozen manifest")
    if any("associated_entities" not in hit for hit in hits):
        raise ValueError(f"{modality}: associated_entities missing from GDC response")
    if any(not isinstance(hit["associated_entities"], list) for hit in hits):
        raise ValueError(f"{modality}: invalid associated_entities type")

    if not snapshot_path.is_file():
        snapshot_path.parent.mkdir(parents=True, exist_ok=True)
        pending_path = snapshot_path.with_suffix(".json.pending")
        if pending_path.exists():
            raise FileExistsError(f"Refusing to overwrite pending snapshot: {pending_path}")
        with pending_path.open("x", encoding="utf-8") as handle:
            json.dump(snapshot, handle, indent=2, ensure_ascii=False)
            handle.write("\n")
        pending_path.replace(snapshot_path)
        print(f"{modality}: saved {project_relative_path(snapshot_path)}")

    association_snapshots[modality] = {
        str(hit.get("file_id") or hit.get("id")): hit for hit in hits
    }
    print(f"{modality}: {len(observed_ids):,} exact-file association records verified")

In [ ]:
# =============================================================================
# Resolve case/sample/aliquot ancestry in initial GDC metadata
# =============================================================================

def aliquot_ancestry(file_record):
    mapping = {}
    for case in file_record.get("cases") or []:
        case_uuid = case.get("case_id")
        case_submitter_id = case.get("submitter_id")
        project_id = (case.get("project") or {}).get("project_id")
        for sample in case.get("samples") or []:
            sample_info = {
                "case_uuid": case_uuid,
                "case_submitter_id": case_submitter_id,
                "project_id": project_id,
                "sample_uuid": sample.get("sample_id"),
                "sample_submitter_id": sample.get("submitter_id"),
                "sample_type": sample.get("sample_type"),
                "tissue_type": sample.get("tissue_type"),
                "tumor_descriptor": sample.get("tumor_descriptor"),
            }
            for portion in sample.get("portions") or []:
                for analyte in portion.get("analytes") or []:
                    for aliquot in analyte.get("aliquots") or []:
                        aliquot_uuid = aliquot.get("aliquot_id")
                        if aliquot_uuid:
                            mapping.setdefault(aliquot_uuid, []).append(sample_info)
    return mapping


def case_annotation(file_record):
    annotations = file_record.get("cases") or []
    return {
        row.get("case_id"): row for row in annotations
        if row.get("case_id") is not None
    }

In [ ]:
# =============================================================================
# Construct direct file–aliquot–sample mappings without case-level expansion
# =============================================================================

file_index_rows = []
associated_entity_rows = []

for modality, spec in SOURCE_SPECS.items():
    for row in manifests[modality].itertuples(index=False):
        source = source_hits[modality][row.id]
        direct = association_snapshots[modality][row.id]
        ancestry = aliquot_ancestry(source)
        case_info = case_annotation(direct)
        entities = direct.get("associated_entities") or []
        resolved = []
        issues = []

        for entity in entities:
            entity_type = entity.get("entity_type")
            entity_uuid = entity.get("entity_id")
            direct_case_uuid = entity.get("case_id")
            parents = ancestry.get(entity_uuid, []) if entity_type == "aliquot" else []
            # The case-level sample list alone is never an acceptable file mapping.
            distinct_parents = {
                (parent.get("case_uuid"), parent.get("sample_uuid")): parent
                for parent in parents
            }
            if entity_type != "aliquot":
                issues.append("non_aliquot_entity")
            elif len(distinct_parents) != 1:
                issues.append("aliquot_hierarchy_unresolved_or_ambiguous")
            else:
                parent = next(iter(distinct_parents.values()))
                if direct_case_uuid and parent["case_uuid"] != direct_case_uuid:
                    issues.append("direct_case_conflict")
                else:
                    resolved.append((entity, parent))

            associated_entity_rows.append({
                "modality": modality,
                "file_id": row.id,
                "entity_type": entity_type,
                "entity_uuid": entity_uuid,
                "entity_submitter_id": entity.get("entity_submitter_id"),
                "direct_case_uuid": direct_case_uuid,
                "n_hierarchy_parent_samples": len(distinct_parents),
            })

        if len(entities) == 1 and len(resolved) == 1 and not issues:
            mapping_status = "unique_aliquot_sample"
            entity, parent = resolved[0]
            case_record = case_info.get(parent["case_uuid"], {})
        elif not entities:
            mapping_status = "missing_direct_association"
            entity, parent, case_record = None, None, {}
        elif len(entities) > 1:
            mapping_status = "multiple_direct_associations"
            entity, parent, case_record = None, None, {}
        else:
            mapping_status = "unresolved_direct_association"
            entity, parent, case_record = None, None, {}

        file_index_rows.append({
            "modality": modality,
            "file_id": row.id,
            "filename": row.filename,
            "size_bytes": int(row.size),
            "provider_md5": row.md5.lower(),
            "workflow_type": spec["workflow_type"],
            "platform": direct.get("platform"),
            "n_direct_entities": len(entities),
            "entity_types": "|".join(sorted({
                str(item.get("entity_type")) for item in entities
            })),
            "mapping_status": mapping_status,
            "mapping_issues": "|".join(sorted(set(issues))),
            "aliquot_uuid": entity.get("entity_id") if entity else None,
            "aliquot_submitter_id": entity.get("entity_submitter_id") if entity else None,
            "case_uuid": parent["case_uuid"] if parent else None,
            "case_submitter_id": parent["case_submitter_id"] if parent else None,
            "sample_uuid": parent["sample_uuid"] if parent else None,
            "sample_submitter_id": parent["sample_submitter_id"] if parent else None,
            "sample_type": parent["sample_type"] if parent else None,
            "tissue_type": parent["tissue_type"] if parent else None,
            "tumor_descriptor": parent["tumor_descriptor"] if parent else None,
            "project_id": parent["project_id"] if parent else None,
            "primary_site": case_record.get("primary_site"),
            "disease_type": case_record.get("disease_type"),
        })

file_index = pd.DataFrame(file_index_rows).sort_values(
    ["modality", "file_id"]
).reset_index(drop=True)
associated_entity_index = pd.DataFrame(associated_entity_rows)
if len(file_index) != sum(len(frame) for frame in manifests.values()):
    raise ValueError("Not all acquisition files are represented in the file index")
if file_index.duplicated(["modality", "file_id"]).any():
    raise ValueError("Duplicated GDC file identity in the audit index")

In [ ]:
# =============================================================================
# Audit direct mapping coverage and unresolved association categories
# =============================================================================

mapping_coverage = (
    file_index.groupby(["modality", "mapping_status"], dropna=False)
    .agg(n_files=("file_id", "size"))
    .reset_index()
    .sort_values(["modality", "mapping_status"])
)
display(mapping_coverage)

print("Direct-entity type distribution:")
if not associated_entity_index.empty:
    display(
        associated_entity_index.groupby(
            ["modality", "entity_type"], dropna=False
        ).size().rename("n_entities").reset_index()
    )
else:
    print("GDC returned no directly associated entities")

In [ ]:
# =============================================================================
# Characterize uniquely resolved sample types and platform availability
# =============================================================================

resolved_file_index = file_index.loc[
    file_index["mapping_status"].eq("unique_aliquot_sample")
].copy()

sample_type_coverage = (
    resolved_file_index.groupby(
        ["modality", "sample_type"], dropna=False
    )
    .agg(n_files=("file_id", "size"), n_samples=("sample_uuid", "nunique"),
         n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["modality", "n_files"], ascending=[True, False])
)
display(sample_type_coverage)

print("Platform availability (file-level):")
display(
    file_index.groupby(["modality", "platform"], dropna=False)
    .size().rename("n_files").reset_index()
)

## Provisional primary-tumor inventory and multi-omic overlap

Only a uniquely resolved direct aliquot → sample mapping can enter the provisional primary-tumor inventory. This **fixed technical rule** does not establish cohort eligibility for a future inferential Phase 8 analysis. Different GDC files may correspond to the same biological sample, and different platforms can have non-comparable feature spaces.

Cross-modality overlap is evaluated separately at case UUID and exact sample UUID resolution; matching by case alone is not called a paired multi-omic sample. Raw case-wide sample lists are not used for these counts.

In [ ]:
# =============================================================================
# Apply prespecified provisional primary-tumor technical flag
# =============================================================================

file_index["candidate_primary_tumor"] = (
    file_index["mapping_status"].eq("unique_aliquot_sample")
    & file_index["sample_type"].eq("Primary Tumor")
)

candidate_primary_files = file_index.loc[
    file_index["candidate_primary_tumor"]
].copy()

display(
    candidate_primary_files.groupby("modality")
    .agg(
        n_files=("file_id", "size"),
        n_samples=("sample_uuid", "nunique"),
        n_cases=("case_uuid", "nunique"),
    )
)
print("No Phase 8 analytic eligibility or lineage/platform comparability is inferred.")

In [ ]:
# =============================================================================
# Characterize file multiplicity within biological samples
# =============================================================================

multiplicity = (
    resolved_file_index.groupby(
        ["modality", "sample_uuid"], dropna=False
    )["file_id"]
    .nunique()
    .rename("n_files")
    .reset_index()
)
display(
    multiplicity.groupby(["modality", "n_files"])
    .size()
    .rename("n_samples")
    .reset_index()
    .sort_values(["modality", "n_files"])
)

In [ ]:
# =============================================================================
# Cross-modality overlap at exact case/sample UUID resolution
# =============================================================================

def shared_identifier_count(table, key, left="rnaseq", right="methylation"):
    lhs = set(table.loc[table["modality"].eq(left), key].dropna())
    rhs = set(table.loc[table["modality"].eq(right), key].dropna())
    return len(lhs & rhs)


overlap_summary = {
    "uniquely_resolved_cases_shared": shared_identifier_count(
        resolved_file_index, "case_uuid"
    ),
    "uniquely_resolved_samples_shared": shared_identifier_count(
        resolved_file_index, "sample_uuid"
    ),
    "provisional_primary_cases_shared": shared_identifier_count(
        candidate_primary_files, "case_uuid"
    ),
    "provisional_primary_samples_shared": shared_identifier_count(
        candidate_primary_files, "sample_uuid"
    ),
}
display(pd.Series(overlap_summary, name="count").to_frame())

print("Case matching is not sample matching; primary-case overlap is not")
print("a claim of biologically paired multi-omic specimens.")

In [ ]:
# =============================================================================
# Audit lineage metadata coverage in the provisional primary-tumor inventory
# =============================================================================

lineage_coverage = (
    candidate_primary_files.groupby(
        ["modality", "primary_site", "disease_type"], dropna=False
    )
    .agg(n_files=("file_id", "size"), n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["modality", "n_files"], ascending=[True, False])
)
display(lineage_coverage)

print("Missing primary-site labels:",
      int(candidate_primary_files["primary_site"].isna().sum()))
print("Missing disease-type labels:",
      int(candidate_primary_files["disease_type"].isna().sum()))

## Persistence and review checkpoint

Outputs are acquisition-audit artifacts only. The original provider manifests, initial metadata JSON, and raw payloads remain unchanged.

- File-level audit and direct-association index: data/interim/metadata/.
- Complete GDC associated-entity snapshots: config/manifests/cptac3/.
- Registry proposal: data/interim/metadata/; **not** applied automatically.

All unresolved and multiply associated files remain visible in the full audit index. Cohort freeze, selection of one file per biological unit, methylation-platform compatibility, harmonization, and program-level validation are deferred pending review. Results may be unfavorable; no post hoc rescue is permitted.

In [ ]:
# =============================================================================
# Persist deterministic Phase 1 CPTAC-3 audit tables
# =============================================================================

AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATHS = {
    "payload_validation": AUDIT_OUTPUT_DIR / "111_cptac3_download_validation.csv",
    "file_index": AUDIT_OUTPUT_DIR / "111_cptac3_file_index.csv",
    "direct_entities": AUDIT_OUTPUT_DIR / "111_cptac3_direct_entities.csv",
    "candidate_primary": AUDIT_OUTPUT_DIR / "111_cptac3_provisional_primary_tumor_files.csv",
}

output_tables = {
    "payload_validation": payload_validation,
    "file_index": file_index,
    "direct_entities": associated_entity_index,
    "candidate_primary": candidate_primary_files,
}

for name, table in output_tables.items():
    table.to_csv(AUDIT_PATHS[name], index=False)
    print(name, project_relative_path(AUDIT_PATHS[name]), len(table))

In [ ]:
# =============================================================================
# Reload and validate persisted audit outputs
# =============================================================================

for name, path in AUDIT_PATHS.items():
    reloaded = pd.read_csv(path, dtype="string")
    expected = output_tables[name]
    if len(reloaded) != len(expected):
        raise ValueError(f"Persisted {name} row count mismatch")
    if list(reloaded.columns) != list(expected.columns):
        raise ValueError(f"Persisted {name} columns differ")
    if name in {"file_index", "payload_validation"}:
        if reloaded[["modality", "file_id"]].duplicated().any():
            raise ValueError(f"Persisted {name} has duplicate file IDs")
    print(f"{name}: {len(reloaded):,} rows / schema PASS")

In [ ]:
# =============================================================================
# Build source and audit artifact fingerprints
# =============================================================================

SOURCE_FINGERPRINTS = {
    modality: {
        "manifest": {
            "path": project_relative_path(spec["manifest"]),
            "sha256": calculate_sha256(spec["manifest"]),
        },
        "metadata": {
            "path": project_relative_path(spec["metadata"]),
            "sha256": source_metadata_sha256[modality],
        },
        "associated_entities": {
            "path": project_relative_path(association_paths[modality]),
            "sha256": calculate_sha256(association_paths[modality]),
        },
        "payload_dir": project_relative_path(spec["payload_dir"]),
    }
    for modality, spec in SOURCE_SPECS.items()
}
AUDIT_FINGERPRINTS = {
    name: {
        "path": project_relative_path(path),
        "sha256": calculate_sha256(path),
        "size_bytes": path.stat().st_size,
        "rows": len(output_tables[name]),
    }
    for name, path in AUDIT_PATHS.items()
}

In [ ]:
# =============================================================================
# Persist a factual acquisition-audit summary (not a cohort freeze)
# =============================================================================

SUMMARY_PATH = AUDIT_OUTPUT_DIR / "111_cptac3_acquisition_audit_summary.json"

audit_summary = {
    "notebook": 111,
    "audit_role": "phase1_source_acquisition_and_technical_eligibility_assessment",
    "cohort_frozen": False,
    "phase8_validation_performed": False,
    "raw_payload_modified": False,
    "full_payload_md5_verified": RUN_FULL_PAYLOAD_MD5,
    "source_fingerprints": SOURCE_FINGERPRINTS,
    "audit_artifacts": AUDIT_FINGERPRINTS,
    "n_files_by_modality": {
        modality: len(manifests[modality]) for modality in SOURCE_SPECS
    },
    "mapping_status_counts": {
        modality: {
            str(status): int(count)
            for status, count in zip(
                frame["mapping_status"],
                frame["n_files"],
            )
        }
        for modality, frame in mapping_coverage.groupby("modality")
    },
    "overlap_at_directly_resolved_unit": overlap_summary,
    "provisional_primary_definition": {
        "mapping_status": "unique_aliquot_sample",
        "sample_type": "Primary Tumor",
    },
    "scientific_limitations": [
        "case-level samples are not file-specific without direct aliquot association",
        "clinical outcome/resistance status is not inferred",
        "case overlap is not exact sample matching",
        "one sample may contribute multiple GDC files",
        "lineage and methylation platform confounding require downstream evaluation",
        "candidate primary-tumor status is not final Phase 8 eligibility",
    ],
}
with SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(audit_summary, handle, indent=2, ensure_ascii=False)
    handle.write("\n")

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    summary_check = json.load(handle)
if summary_check["n_files_by_modality"] != audit_summary["n_files_by_modality"]:
    raise ValueError("Persisted acquisition summary is inconsistent")

print(project_relative_path(SUMMARY_PATH))

In [ ]:
# =============================================================================
# Build a manifest-managed raw-data registry proposal (no registry mutation)
# =============================================================================

raw_registry = load_raw_data_registry()
if "cptac3" in raw_registry:
    raise ValueError(
        "CPTAC-3 is already registered; reconcile with existing entry manually"
    )

cptac3_entry = {
    "source_database": "Genomic Data Commons (GDC)",
    "provider": "National Cancer Institute",
    "download_page_url": "https://portal.gdc.cancer.gov/repository",
    "retrieval_method": "GDC Files API exact-manifest acquisition and file-level metadata audit",
    "cohorts": {},
}
for modality, spec in SOURCE_SPECS.items():
    cptac3_entry["cohorts"][modality] = {
        "provenance_mode": "manifest_managed",
        "status": "acquired_not_used",
        "audit_exclusion_reason": (
            "GDC manifest UUID, filename, size, MD5 and notebook 111 "
            "local validation govern individual payload identity; "
            "individual payload SHA256 is not separately registered."
        ),
        "release": "GDC CPTAC-3 live open-file query snapshot; data release not asserted",
        "canonical_dir": "config/manifests/cptac3",
        "payload_dir": project_relative_path(spec["payload_dir"]),
        "filters": {
            "project_id": "CPTAC-3",
            "access": "open",
            "data_category": spec["data_category"],
            "data_type": spec["data_type"],
            "workflow_type": spec["workflow_type"],
        },
        "files": {
            spec["manifest"].name: {"role": "manifest"},
            spec["metadata"].name: {"role": "metadata_json"},
            association_paths[modality].name: {"role": "associated_entities_snapshot"},
        },
    }

proposal = deepcopy(raw_registry)
proposal["cptac3"] = cptac3_entry
validate_raw_data_registry(proposal)

PROPOSAL_PATH = AUDIT_OUTPUT_DIR / "111_cptac3_raw_registry_proposal.json"
with PROPOSAL_PATH.open("w", encoding="utf-8") as handle:
    json.dump(cptac3_entry, handle, indent=2, ensure_ascii=False)
    handle.write("\n")

print("Proposed CPTAC-3 entry:", project_relative_path(PROPOSAL_PATH))
print("config/raw_data_registry.json remains unchanged")

## Completion status — review required before freeze

If all cells run successfully, CPTAC-3 local file integrity, provider-file identity, direct biospecimen association coverage, provisional primary-tumor availability, and case/sample overlaps have been audited. The exact numerical findings are in the generated tables and summary, not predetermined in this notebook.

**Not yet closed:** review the complete unresolved-association categories, sample and file multiplicity, lineage/platform metadata completeness, and the provisional cohort universe. The registry proposal should be accepted and applied in a controlled repository change only after reviewing those outputs. Phase 8 biological validation remains out of scope.

This notebook does not repair missing or ambiguous biospecimen links, treat normal tissue as tumor, interpret cases as independent files, or claim external program validation.